In [16]:
import pandas as pd
import  numpy as np

from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV

from sklearn.metrics import r2_score

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor

In [2]:
df = pd.read_csv("Boston.csv")

In [3]:
df.head()

,Unnamed: 0,crim,zn,indus,chas,nox,rm,age,dis,rad,tax,ptratio,black,lstat,medv
0,1,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296,15.3,396.90,4.98,24.0
1,2,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242,17.8,396.90,9.14,21.6
2,3,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242,17.8,392.83,4.03,34.7
3,4,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222,18.7,394.63,2.94,33.4
4,5,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222,18.7,396.90,5.33,36.2


In [4]:
df = df.iloc[:,1:]

In [5]:
df.head()

,crim,zn,indus,chas,nox,rm,age,dis,rad,tax,ptratio,black,lstat,medv
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222,18.7,396.90,5.33,36.2


In [6]:
X = df.iloc[:,0:13]
y = df.iloc[:,13]

In [9]:
X.columns

Index(['crim', 'zn', 'indus', 'chas', 'nox', 'rm', 'age', 'dis', 'rad', 'tax',
       'ptratio', 'black', 'lstat'],
      dtype='object')

In [11]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=23)

In [12]:
print("Train/Test sets sizes", X_train.shape, X_test.shape, y_train.shape, y_test.shape)

Train/Test sets sizes (404, 13) (102, 13) (404,) (102,)


# Now we will separately train our models 

In [17]:
lr = LinearRegression()
dt = DecisionTreeRegressor()
svr = SVR()
knn = KNeighborsRegressor()

In [18]:
lr.fit(X_train, y_train)
dt.fit(X_train, y_train)
svr.fit(X_train, y_train)
knn.fit(X_train, y_train)

KNeighborsRegressor()

In [22]:
y_pred1 = lr.predict(X_test)
y_pred2 = dt.predict(X_test)
y_pred3 = svr.predict(X_test)
y_pred4 = knn.predict(X_test)

In [23]:
print("R2 score of LR", r2_score(y_test, y_pred1))
print("R2 score of DT", r2_score(y_test, y_pred2))
print("R2 score of SVR", r2_score(y_test, y_pred3))
print("R2 score of KNN", r2_score(y_test, y_pred4))

R2 score of LR 0.7451430642919583
R2 score of DT 0.7311627051171192
R2 score of SVR 0.17350879361938487
R2 score of KNN 0.5222138859051638


In [24]:
# As individual linear Refression is performing good as we can see above

# Now we will check the same process with the help of Bagging Regressor 

In [25]:
from sklearn.ensemble import BaggingRegressor

In [26]:
bag = BaggingRegressor()
bag.fit(X_train, y_train)

BaggingRegressor()

In [27]:
y_pred = bag.predict(X_test)

In [31]:
r2_score(y_test, y_pred) 

0.7893878674670718

In [33]:
print("R2 score on trainig data: ", bag.score(X_train, y_train))
print("R2 score on testing data: ", bag.score(X_test, y_test))

R2 score on trainig data:  0.978808491275226
R2 score on testing data:  0.7893878674670718


# Now we will do hyperparameter tuning to find the best parameters

In [38]:
%time

n_features = df.shape[1]
n_samples = df.shape[0]

params = {
    "estimator" : [LinearRegression(), None, KNeighborsRegressor()],
    "n_estimators" : [20,50, 100],
    "max_samples" : [0.5, 1.0],
    "max_features" : [0.5, 1.0],
    "bootstrap" : [True, False],
    "bootstrap_features" : [True, False]
}

bagging_refressor_grid = GridSearchCV(BaggingRegressor(random_state=1, n_jobs=-1), param_grid=params, cv=3, n_jobs=-1, verbose=1)
bagging_refressor_grid.fit(X_train, y_train)

print("Train R2 score: ", bagging_refressor_grid.best_estimator_.score(X_train, y_train))
print("Test R2 score: ", bagging_refressor_grid.best_estimator_.score(X_test, y_test))
print("Best R2 score through Grid search: ", bagging_refressor_grid.best_score_)
print("Best parameters: ", bagging_refressor_grid.best_params_)

CPU times: total: 0 ns
Wall time: 0 ns
Fitting 3 folds for each of 144 candidates, totalling 432 fits
Train R2 score:  0.9843004600450516
Test R2 score:  0.8378096356473124
Best R2 score through Grid search:  0.8723274253954026
Best parameters:  {'bootstrap': True, 'bootstrap_features': False, 'estimator': None, 'max_features': 1.0, 'max_samples': 1.0, 'n_estimators': 100}


In [48]:
bag1 = BaggingRegressor(estimator=None, max_samples=1.0, max_features=1.0, n_estimators=100, bootstrap=True, bootstrap_features=False)

In [49]:
bag1.fit(X_train, y_train)

BaggingRegressor(n_estimators=100)

In [50]:
y_pred = bag1.predict(X_test)

In [54]:
print("R2 score after using Grid searchCV: ", np.round(r2_score(y_test, y_pred),2) * 100)

R2 score after using Grid searchCV:  84.0


# Thats impressive